# INTERVAL: process local CSV files

Join existing Places and Submissions and export separate CSVs for Places with
one Submission and Places with multiple Submissions. This workflow keeps only
matching records (an inner join); it does not download data or select the latest
Submission. See [the detailed guide](../docs/INTERVAL_process_data.md).

Install `pandas` in your notebook kernel environment before running. Processing
logic lives in `INTERVAL_process_data.py`, which also provides a command-line interface.
Restart the kernel and run all cells after changing that module.


In [1]:
from pathlib import Path
import sys

# Open this notebook from the repository root or src/.
HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "src" else HERE
if not (ROOT / "src" / "INTERVAL_process_data.py").is_file():
    raise RuntimeError("Start Jupyter from the repository root or src/ directory")
sys.path.insert(0, str(ROOT / "src"))

from project_paths import project_root
from INTERVAL_process_data import read_csv, process_data, write_outputs

ROOT = project_root()


## 1. Choose input and output paths

Use a matching pair of raw INTERVAL `places.csv` and `submissions.csv` exports.
The defaults below preserve the original locations. For a synthetic demonstration,
set `INPUT_DIR = ROOT / "tests" / "fixtures" / "interval_local"` and use a separate
output directory, for example `ROOT / "outputs" / "interval_local_demo"`.


In [2]:
INPUT_DIR = ROOT / "outputs" / "csv_files"
OUTPUT_DIR = ROOT / "outputs" / "csv_files"
PLACES_PATH = INPUT_DIR / "places.csv"
SUBMISSIONS_PATH = INPUT_DIR / "submissions.csv"


## 2. Load and inspect

Values are read as text to preserve IDs and literal responses such as `NA`.
Missing files raise an error immediately. Correct the path before continuing.


In [3]:
places_df = read_csv(PLACES_PATH)
submissions_df = read_csv(SUBMISSIONS_PATH)
print(f"Places: {len(places_df)}; Submissions: {len(submissions_df)}")
places_df.head()


Places: 120; Submissions: 147


,Id,unique_id,location_type,tree_privacy,NTM_ID,tree_height,avg_tree_height,canopy_perimeter,canopy_area,STD,geometry_x,geometry_y,dataset_id,dataset_name,url,submitter_name,submitter_id,submitter_user
0,46065,,no-tree,Public,NTM_P_718728_71853987_72976050,3.7,3.04,11,5.38,0.53,-6.221412448202982,53.304851616712924,342,Building 71,https://interval.cartospot.com/map/building-71...,INTERVAL Project,51,INTERVAL Team
1,46039,,completed,Public,NTM_P_718728_71854262_72978150,14.86,10.33,79,166.69,3.84,-6.221376836323216,53.30504299586885,342,Building 71,https://interval.cartospot.com/map/building-71...,INTERVAL Project,51,INTERVAL Team
2,46089,,completed,Public,NTM_P_718728_71865500_72980300,13.72,10.97,45,70.13,2.72,-6.219665634144017,53.305210329703286,342,Building 71,https://interval.cartospot.com/map/building-71...,INTERVAL Project,51,INTERVAL Team
3,46123,,completed,Public,,,,,,,-6.219888478517532,53.30523052864868,342,Building 71,https://interval.cartospot.com/map/building-71...,,,
4,46121,,completed,Public,,,,,,,-6.219908595085144,53.305270998300024,342,Building 71,https://interval.cartospot.com/map/building-71...,,,


In [4]:
submissions_df.head()


,Id,location_id,location_unique_id,does_a_tree_exist_here,species,other_species,tree_type,health,circumference,dbh,notes,image_url,submitter_name,submitter_id,submitter_user
0,47134,46039,,Yes,Sycamore Maple,,Deciduous,Good,296,94.22,,https://community-georeports.fra1.cdn.digitalo...,Eve and Joan,,
1,47133,46039,,Yes,Sycamore Maple,,Deciduous,Good,296,94.22,,,Eve and Joan,,
2,46136,46039,,Yes,Other,Platanus,Deciduous,Good,278,88.49,,https://community-georeports.fra1.cdn.digitalo...,Khadeeja,,
3,47131,46089,,Yes,Grey Alder,,Deciduous,Very Good,197,62.707,,https://community-georeports.fra1.cdn.digitalo...,9,,
4,47123,46089,,Yes,Pin Oak,,Deciduous,Very Good,97,30.876,,https://community-georeports.fra1.cdn.digitalo...,9,,


## 3. Validate, join and split

Place and Submission IDs must be unique within their source tables. Duplicate
IDs or missing required columns stop processing with a descriptive error.
The summary counts unmatched records excluded from the inner join.


In [5]:
result = process_data(places_df, submissions_df)
single_submission_df = result.single
multiple_submissions_df = result.multiple
result.summary


{'input_places': 120,
 'input_submissions': 147,
 'matched_places': 97,
 'matched_submissions': 147,
 'places_without_submissions': 23,
 'unmatched_submissions': 0,
 'single_submission_places': 63,
 'multiple_submission_places': 34,
 'multiple_submission_rows': 84}

In [6]:
single_submission_df.head()


,Id_x,location_type,tree_height,geometry_x,geometry_y,dataset_name,url,Id_y,location_id,location_unique_id,...,tree_type,health,circumference,dbh,notes,image_url,submitter_name,submitter_id,submitter_user,submission_count
8,46123,completed,,-6.219888478517532,53.30523052864868,Building 71,https://interval.cartospot.com/map/building-71...,47130,46123,,...,Deciduous,Excellent,45,14.324,,https://community-georeports.fra1.cdn.digitalo...,Group 12,,,1
21,46094,completed,16.68,-6.219623346611183,53.3050388897329,Building 71,https://interval.cartospot.com/map/building-71...,47124,46094,,...,Deciduous,Very Good,110,35.014,,https://community-georeports.fra1.cdn.digitalo...,Group 6,,,1
29,46067,completed,11.06,-6.2194958432349745,53.30496650497716,Building 71,https://interval.cartospot.com/map/building-71...,47116,46067,,...,Deciduous,Very Good,140,44.563,,https://community-georeports.fra1.cdn.digitalo...,Group 6,,,1
48,46122,image-needed,,-6.220056116580964,53.30507666348555,Building 71,https://interval.cartospot.com/map/building-71...,46139,46122,,...,Deciduous,,37,11.777,,,Kayla Dunleavy,,,1
49,46025,image-needed,9.77,-6.22008307018023,53.305482918719896,Building 71,https://interval.cartospot.com/map/building-71...,46157,46025,,...,Deciduous,Good,250,79.577,,,Cormac Byrne,,,1


In [7]:
multiple_submissions_df.head()


,Id_x,location_type,tree_height,geometry_x,geometry_y,dataset_name,url,Id_y,location_id,location_unique_id,...,tree_type,health,circumference,dbh,notes,image_url,submitter_name,submitter_id,submitter_user,submission_count
0,46039,completed,14.86,-6.221376836323216,53.30504299586885,Building 71,https://interval.cartospot.com/map/building-71...,47134,46039,,...,Deciduous,Good,296,94.22,,https://community-georeports.fra1.cdn.digitalo...,Eve and Joan,,,3
1,46039,completed,14.86,-6.221376836323216,53.30504299586885,Building 71,https://interval.cartospot.com/map/building-71...,47133,46039,,...,Deciduous,Good,296,94.22,,,Eve and Joan,,,3
2,46039,completed,14.86,-6.221376836323216,53.30504299586885,Building 71,https://interval.cartospot.com/map/building-71...,46136,46039,,...,Deciduous,Good,278,88.49,,https://community-georeports.fra1.cdn.digitalo...,Khadeeja,,,3
3,46089,completed,13.72,-6.219665634144017,53.305210329703286,Building 71,https://interval.cartospot.com/map/building-71...,47131,46089,,...,Deciduous,Very Good,197,62.707,,https://community-georeports.fra1.cdn.digitalo...,9,,,5
4,46089,completed,13.72,-6.219665634144017,53.305210329703286,Building 71,https://interval.cartospot.com/map/building-71...,47123,46089,,...,Deciduous,Very Good,97,30.876,,https://community-georeports.fra1.cdn.digitalo...,9,,,5


## 4. Export

Each output row represents a Submission. `Id_x` is the Place ID, `Id_y` is the
Submission ID, and `submission_count` is the number of Submissions for that Place.
Files with these output names are replaced on rerun. Empty groups still have CSV headers.


In [ ]:
output_paths = write_outputs(result, OUTPUT_DIR, (PLACES_PATH, SUBMISSIONS_PATH))
for group, path in output_paths.items():
    print(f"{group}: {path}")
